In [1]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

# Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
job_array=False;job_adjust=0
ocean_fraction=2/8
job_array=False

# dx = 1 km; Np = 1M; Nt = 5 min
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_5min.nc') #***
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_5min_1e6.nc') #***
res='1km';t_res='5min'
Np_str='1e6'


# # dx = 1km; Np = 50M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
# res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 100M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_100M.nc') #***
# res='1km'; t_res='1min'; Np_str='100e6'

# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# # # parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# # # parcel=parcel.isel(time=np.arange(0,400+1))
# res='250m'

In [2]:
times=data['time'].values/(1e9 * 60); times=times.astype(float);
minutes=1/times[1] #1 / minutes per timestep = timesteps per minute
job_array=False;index_adjust=0
ocean_fraction=2/8

In [3]:
def check_memory():
    import sys
    ipython_vars = ["In", "Out", "exit", "quit", "get_ipython", "ipython_vars"]
    print("Top 10 objects with highest memory usage")
    # Get a sorted list of the objects and their sizes
    mem = {
        key: round(value/1e6,2)
        for key, value in sorted(
            [
                (x, sys.getsizeof(globals().get(x)))
                for x in globals()
                if not x.startswith("_") and x not in sys.modules and x not in ipython_vars
            ],
            key=lambda x: x[1],
            reverse=True)[:10]
    }
    print({key:f"{value} MB" for key,value in mem.items()})
    print(f"\n{round(sum(mem.values()),2)/1000} GB in use overall")

In [4]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(PlottingFunctions, inspect.isfunction)]
# functions

In [30]:
#JOB ARRAY SETUP
job_array=True
if job_array==True:

    num_jobs=60 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
    total_elements=len(parcel['xh']) #total num of variables

    if num_jobs >= total_elements:
        raise ValueError("Number of jobs cannot be greater than or equal to total elements.")
    
    job_range = total_elements // num_jobs  # Base size for each chunk
    remaining = total_elements % num_jobs   # Number of chunks with 1 extra 
    
    # Function to compute the start and end for each job_id
    def get_job_range(job_id, num_jobs):
        job_id-=1
        # Add one extra element to the first 'remaining' chunks
        start_job = job_id * job_range + min(job_id, remaining)
        end_job = start_job + job_range + (1 if job_id < remaining else 0)
    
        if job_id == num_jobs - 1: 
            end_job = total_elements #- 1
        return start_job, end_job
    # def job_testing():
    #     #TESTING
    #     start=[];end=[]
    #     for job_id in range(1,num_jobs+1):
    #         start_job, end_job = get_job_range(job_id)
    #         print(start_job,end_job)
    #         start.append(start_job)
    #         end.append(end_job)
    #     print(np.all(start!=end))
    #     print(len(np.unique(start))==len(start))
    #     print(len(np.unique(end))==len(end))
    # job_testing()
    
    job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    if job_id==0: job_id=2
    start_job, end_job = get_job_range(job_id, num_jobs)
    index_adjust=start_job
    print(f'start_job = {start_job}, end_job = {end_job}')

start_job = 16667, end_job = 33334


In [ ]:
# Reading Back Data Later
##############
def make_data_dict(var_names,read_type):
    if read_type=='h5py':
        with h5py.File(in_file, 'r') as f:
            data_dict = {var_name: f[var_name][:,start_job:end_job] for var_name in var_names}
            
    elif read_type=='xarray':
        in_data = xr.open_dataset(
            in_file,
            engine='h5netcdf',
            phony_dims='sort',
            chunks={'phony_dim_0': 100, 'phony_dim_1': 1_000_000} 
        )
        data_dict = {k: in_data[k][:,start_job:end_job].compute().data for k in var_names}
    return data_dict

# read_type='xarray'
read_type='h5py'

In [ ]:
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_5min.h5'
# in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_1min_50M.h5'

var_names = ['A_g', 'A_c', 'W', 'QCQI', 'Z', 'Y', 'X', 'z']
data_dict = make_data_dict(var_names,read_type)
A_g, A_c, W, QCQI, Z, Y, X, parcel_z = (data_dict[k] for k in var_names)

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)
check_memory()

In [ ]:
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'VMF_binary_array_{res}_{Np_str}_5min.h5'
# in_file=dir2+f'VMF_binary_array_{res}_{Np_str}_1min_50M.h5'

var_names = ['VMF_G','VMF_C']
data_dict = make_data_dict(var_names,read_type)
VMF_G, VMF_C = (data_dict[k] for k in var_names)

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)
check_memory()

In [33]:
#TRACKED TRAJECTORY ENTRAINMENT/DETRAINMENT
################################################################

In [ ]:
########################
#READING BACK IN
def LoadFinalData(in_file):
    dict = {}
    with h5py.File(in_file, 'r') as f:
        for key in f.keys():
            dict[key] = f[key][:]
    return dict

def LoadAllCloudBase():
    dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/'
    in_file = dir2 + f"all_cloudbase_{res}_{t_res}_{Np_str}.pkl"
    with open(in_file, 'rb') as f:
        all_cloudbase = pickle.load(f)
    return(all_cloudbase)
min_all_cloudbase=np.nanmin(LoadAllCloudBase())
print(f"Minimum Cloudbase is: {min_all_cloudbase}\n")

dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/'
in_file=dir2+f"parcel_tracking_SUBSET_{res}_{t_res}_{Np_str}"
final_dict=LoadFinalData(in_file)


#DYNAMICALLY CREATING VARIABLES
for key, value in final_dict.items():
    globals()[key] = value

# #DYNAMICALLY PRINTING VARIABLE SIZES
# for key in final_dict:
#     print(f"{key} has {final_dict[key].shape[0]} parcels")

# PRINTING VARIABLE SIZES (ONE BY ONE)
print(f'ALL: {len(CL_ALL_out_arr)} CL parcels and {len(nonCL_ALL_out_arr)} nonCL parcels')
print(f'SHALLOW: {len(CL_SHALLOW_out_arr)} CL parcels and {len(nonCL_SHALLOW_out_arr)} nonCL parcels')
print(f'DEEP: {len(CL_DEEP_out_arr)} CL parcels and {len(nonCL_DEEP_out_arr)} nonCL parcels')
print('\n')
print(f'ALL: {len(SBZ_ALL_out_arr)} SBZ parcels and {len(nonSBZ_ALL_out_arr)} nonSBZ parcels')
print(f'SHALLOW: {len(SBZ_SHALLOW_out_arr)} SBZ parcels and {len(nonSBZ_SHALLOW_out_arr)} nonSBZ parcels')
print(f'DEEP: {len(SBZ_DEEP_out_arr)} SBZ parcels and {len(nonSBZ_DEEP_out_arr)} nonSBZ parcels')
print('\n')
print(f'ALL: {len(ColdPool_ALL_out_arr)} ColdPool parcels')
print(f'SHALLOW: {len(ColdPool_SHALLOW_out_arr)} ColdPool parcels')
print(f'DEEP: {len(ColdPool_DEEP_out_arr)} ColdPool parcels')


#APPLYING JOB ARRAY
if "job_array" in globals():
    print('APPLYING JOB ARRAY')
    def job_filter(arr):
        return arr[(arr[:,0]>=start_job)&(arr[:,0]<end_job)]
    for name in [
        'CL_ALL_out_arr', 'CL_ALL_out_arr',
        'CL_SHALLOW_out_arr', 'CL_SHALLOW_out_arr',
        'CL_DEEP_out_arr', 'nonCL_DEEP_out_arr',
        'SBZ_ALL_out_arr', 'nonSBZ_ALL_out_arr',
        'SBZ_SHALLOW_out_arr', 'nonSBZ_SHALLOW_out_arr',
        'SBZ_DEEP_out_arr', 'nonSBZ_DEEP_out_arr',
        'ColdPool_ALL_out_arr', 'ColdPool_SHALLOW_out_arr', 'ColdPool_DEEP_out_arr'
    ]:
        globals()[name] = job_filter(globals()[name])

In [41]:
##########################################################################################
#(ALL, SHALLOW, DEEP) CL vs NonCL Tracked VMF Profiles

In [42]:
#MAKING ENTRAINMENT PROFILE ARRAY FUNCTION
    
def CLTrackedVMFProfile(type,type2):

    if type2=='general':
        profile_array_VMF=VMF_G.copy() 
    elif type2=='cloudy':
        profile_array_VMF=VMF_C.copy() 

    if type=='all':
        out_arr=ALL_out_arr.copy()
        # after_array=ALL_out_after_array
    elif type=='shallow':
        out_arr=SHALLOW_out_arr.copy()
        # after_array=SHALLOW_out_after_array
    elif type=='deep':
        out_arr=DEEP_out_arr.copy()
        # after_array=DEEP_out_after_array
     
    zhs=data['zh'].values
    VMF_profile=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    VMF_profile[:,2]=zhs

    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust] #JOBARRAY
        ys=Y[ts,p-index_adjust] #JOBARRAY
        xs=X[ts,p-index_adjust] #JOBARRAY
        
        vars_VMF=profile_array_VMF[ts,p-index_adjust] #JOBARRAY
        np.add.at(VMF_profile[:, 0], zs, vars_VMF)
        np.add.at(VMF_profile[:, 1], zs, 1)

    
    print('done\n')
    return VMF_profile

In [43]:
# type2='general'
type2='cloudy'
CL_ALL_VMF = CLTrackedVMFProfile(type='all',type2=type2)
CL_SHALLOW_VMF = CLTrackedVMFProfile(type='shallow',type2=type2)
CL_DEEP_VMF = CLTrackedVMFProfile(type='deep',type2=type2)

#SAVING
import h5py
filePath=dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
filePath+=f"CL_tracked_profiles_VMF_{res}_{t_res}_{Np_str}_{job_id}.h5"
with h5py.File(filePath, 'w') as h5f:
    h5f.create_dataset('CL_ALL_VMF', data=CL_ALL_VMF)
    h5f.create_dataset('CL_SHALLOW_VMF', data=CL_SHALLOW_VMF)
    h5f.create_dataset('CL_DEEP_VMF', data=CL_DEEP_VMF)

done

done

done



In [44]:
#MAKING ENTRAINMENT PROFILE ARRAY FUNCTION
    
def nonCLTrackedVMFProfile(type,type2):

    if type2=='general':
        profile_array_VMF=VMF_G.copy() 
    elif type2=='cloudy':
        profile_array_VMF=VMF_C.copy() 

    if type=='all':
        out_arr=ALL_save_arr
        # after_array=ALL_save_after_array
    elif type=='shallow':
        out_arr=SHALLOW_save_arr
        # after_array=SHALLOW_save_after_array
    elif type=='deep':
        out_arr=DEEP_save_arr
        # after_array=DEEP_save_after_array
     
    zhs=data['zh'].values
    VMF_profile=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    VMF_profile[:,2]=zhs

    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust] #JOBARRAY
        ys=Y[ts,p-index_adjust] #JOBARRAY
        xs=X[ts,p-index_adjust] #JOBARRAY
        
        vars_VMF=profile_array_VMF[ts,p-index_adjust] #JOBARRAY
        np.add.at(VMF_profile[:, 0], zs, vars_VMF)
        np.add.at(VMF_profile[:, 1], zs, 1)

    
    print('done\n')
    return VMF_profile

In [45]:
# type2='general'
type2='cloudy'
nonCL_ALL_VMF = nonCLTrackedVMFProfile(type='all',type2=type2)
nonCL_SHALLOW_VMF = nonCLTrackedVMFProfile(type='shallow',type2=type2)
nonCL_DEEP_VMF = nonCLTrackedVMFProfile(type='deep',type2=type2)

#SAVING
import h5py
filePath=dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
filePath+=f"nonCL_tracked_profiles_VMF_{res}_{t_res}_{Np_str}_{job_id}.h5"
with h5py.File(filePath, 'w') as h5f:
    h5f.create_dataset('nonCL_ALL_VMF', data=nonCL_ALL_VMF)
    h5f.create_dataset('nonCL_SHALLOW_VMF', data=nonCL_SHALLOW_VMF)
    h5f.create_dataset('nonCL_DEEP_VMF', data=nonCL_DEEP_VMF)

done

done

done



In [9]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [ ]:
if recombine==True:
    def get_profiles(job_id):
        # Define file path
        input_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
        input_file+=f"CL_tracked_profiles_VMF_{res}_{t_res}_{Np_str}_{job_id}.h5"
        
        # Load CL-tracked profiles and store them in a dictionary
        data_dict = {}
        
        with h5py.File(input_file, 'r') as h5f:
            for key in h5f.keys():
                data_dict[key] = h5f[key][:]  # Load each dataset into the dictionary
        return data_dict
    
    job_id=1
    dict1=get_profiles(job_id)
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
        dict2=get_profiles(job_id)
        for var in dict2:
            dict1[var][:,0:1+1]+=dict2[var][:,0:1+1]
    
    
    #SAVING INTO FINAL FORM
    output_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'+f"CL_tracked_profiles_VMF_{type2}.h5"
    with h5py.File(output_file, 'w') as f:
        for var in dict1:
            profile_var = dict1[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

In [88]:
if recombine==True:
    def get_profiles(job_id):
        # Define file path
        input_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
        input_file+=f"nonCL_tracked_profiles_VMF_{res}_{t_res}_{Np_str}_{job_id}.h5"
        
        # Load CL-tracked profiles and store them in a dictionary
        data_dict = {}
        
        with h5py.File(input_file, 'r') as h5f:
            for key in h5f.keys():
                data_dict[key] = h5f[key][:]  # Load each dataset into the dictionary
        return data_dict
    
    job_id=1
    dict1=get_profiles(job_id)
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
        dict2=get_profiles(job_id)
        for var in dict2:
            dict1[var][:,0:1+1]+=dict2[var][:,0:1+1]
    
    
    #SAVING INTO FINAL FORM
    output_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'+f"nonCL_tracked_profiles_VMF_{type2}.h5"
    with h5py.File(output_file, 'w') as f:
        for var in dict1:
            profile_var = dict1[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

job_id = 10
job_id = 20
job_id = 30
job_id = 40
job_id = 50
job_id = 60


In [1]:
# #********************************

# if plotting==True:
#     xlims=[] #FIXING XAXIS
#     import matplotlib.pyplot as plt
#     import matplotlib.gridspec as gridspec
    
#     # List of profile arrays and their corresponding labels and x-axis titles
#     profiles = [
#         (CL_ALL_VMF, CL_SHALLOW_VMF, CL_DEEP_VMF, 'VMF', 'CL'),
#         (nonCL_ALL_VMF, nonCL_SHALLOW_VMF, nonCL_DEEP_VMF, 'VMF', 'nonCL')
#     ]
    
    
#     # Set up the 2x3 gridspec
#     fig = plt.figure(figsize=(12, 8))
#     gs = gridspec.GridSpec(1,2, figure=fig)
    
#     # Loop through the profiles and plot them on subplots
#     for i, (ALL_profile_array, SHALLOW_profile_array, DEEP_profile_array, xlabel, CLlabel) in enumerate(profiles):
#         ax = fig.add_subplot(gs[i])
        
#         # Compute the averaged profile
#         ALL_profile = averaged_profiles(ALL_profile_array)
#         SHALLOW_profile = averaged_profiles(SHALLOW_profile_array)
#         DEEP_profile = averaged_profiles(DEEP_profile_array)
    
#         # #TESTING
#         # ##############################
#         # Nx=512;Ny=200;Nt=133;
#         # ALL_profile[:,0]/=(Nx*Ny*Nt)
#         # SHALLOW_profile[:,0]/=(Nx*Ny*Nt)
#         # DEEP_profile[:,0]/=(Nx*Ny*Nt)
#         # ##############################
#         # #TESTING
        
#         # Plot the profile
#         ax.plot(ALL_profile[:, 0], ALL_profile[:, -1],color='black',label='ALL')
#         ax.plot(SHALLOW_profile[:, 0], SHALLOW_profile[:, -1],color='green',label='SHALLOW')
#         ax.plot(DEEP_profile[:, 0], DEEP_profile[:, -1],color='blue',label='DEEP')
#         ax.set_xlabel(xlabel)
#         ax.set_title(CLlabel)
#         ax.set_ylabel('z (km)')
#         ax.grid(True)
        
    
#         #LEGEND
#         ax.legend()
#         # legend_ax = fig.add_subplot(gs[0, 2])  # Use the (2, 3) grid slot for the legend
#         # legend_ax.axis("off")  # Hide axes for the legend box
#         # legend_ax.legend(*ax.get_legend_handles_labels(), loc='center', frameon=False)
    
#     axs = fig.get_axes()
#     ax1,ax2=axs
#     fix_x_limits([ax1,ax2])
#     fix_y_limits([ax1,ax2])
    
#     #ACCESSORIES
#     plt.suptitle(f'CL vs nonCL (ALL, SHALLOW < 4 km, DEEP > 6 km) Vertical Profiles for \n {type2.title()} Updraft VMF from Tracked Lagrangian Parcels')
#     plt.tight_layout()
    
    
#     #MEAN CLOUD BASE
#     axs = fig.get_axes()
#     ax1, ax2= axs
#     for axis in [ax1,ax2]:
#         axis.axhline(all_cloudbase,color='purple',linestyle='dashed')
    
#     #SCIENTIFIC NOTATION
#     apply_scientific_notation([ax1,ax2])

In [ ]:
if plotting==True:
    #REDUCING FOR CONFERENCE FIGURE
    import matplotlib.pyplot as plt
    
    # List of profile arrays and their corresponding labels and x-axis titles
    profiles = [
        (CL_ALL_VMF, CL_SHALLOW_VMF, CL_DEEP_VMF, 'VMF', 'CL'),
        (nonCL_ALL_VMF, nonCL_SHALLOW_VMF, nonCL_DEEP_VMF, 'VMF', 'nonCL')
    ]
    
    cutoff_height=100
    # cutoff_height=7.5
    # cutoff_height=2
    z_cutoff=np.where(CL_ALL_VMF[:,2]<=cutoff_height)[0][-1] #CUT BELOW CERTAIN Z
    
    # Set up the figure
    fig = plt.figure(figsize=(8, 6))
    
    # Create a single axis for all profiles
    ax = fig.add_subplot(111)
    
    # Loop through profiles for CL and nonCL
    for i, (ALL_profile_array, SHALLOW_profile_array, DEEP_profile_array, xlabel, CLlabel) in enumerate(profiles):
        # Compute the averaged profile for ALL
        ALL_profile = averaged_profiles(ALL_profile_array)
        SHALLOW_profile = averaged_profiles(SHALLOW_profile_array)
        DEEP_profile = averaged_profiles(DEEP_profile_array)
    
        #CUT BELOW CERTAIN Z
        ALL_profile=ALL_profile[:z_cutoff]
        SHALLOW_profile=SHALLOW_profile[:z_cutoff]
        DEEP_profile=DEEP_profile[:z_cutoff]
    
        # Plot the profile for ALL (CL and nonCL) on top of each other
        if CLlabel == 'CL':
            ax.plot(ALL_profile[:, 0], ALL_profile[:, -1], color='black', label='ALL (CL)')
            ax.plot(SHALLOW_profile[:, 0], SHALLOW_profile[:, -1], color='green', label='SHALLOW (CL)')
            ax.plot(DEEP_profile[:, 0], DEEP_profile[:, -1], color='blue', label='DEEP (CL)')
        else:
            ax.plot(ALL_profile[:, 0], ALL_profile[:, -1], linestyle='dashed', color='black', label='ALL (nonCL)')
            ax.plot(SHALLOW_profile[:, 0], SHALLOW_profile[:, -1], linestyle='dashed', color='green', label='SHALLOW (nonCL)')
            ax.plot(DEEP_profile[:, 0], DEEP_profile[:, -1], linestyle='dashed', color='blue', label='DEEP (nonCL)')
    
        apply_scientific_notation([ax])
    
    # Add labels and title
    ax.set_xlabel('VMF')  # Replace with actual x-axis label
    ax.set_ylabel('z (km)')
    ax.grid(True)
    
    ax.axvline(0,linestyle='dashed',color='grey',zorder=-10)
    
    # Show legend
    ax.legend()
    
    # Set up the main title and adjust layout
    plt.suptitle(f'CL vs nonCL (ALL, SHALLOW < 4 km, DEEP > 6 km) Vertical Profiles for \n {type2.title()} Updraft VMF from Tracked Lagrangian Parcels')
    plt.tight_layout()
    
    # Show plot
    plt.show()
    


In [55]:
##########################################################################################
#SBZ vs nonSBZ Tracked Entrainment Profiles

In [60]:
#MAKING ENTRAINMENT PROFILE ARRAY FUNCTION

def averaged_profiles(profile):
    out_var = profile[(profile[:, 1] != 0)]  # gets rid of rows that have no data
    # out_var = np.array([out_var[:, 0] / out_var[:, 1], out_var[:, 2]]).T  # divides the data column by the counter column
    
    Nt=len(data['time']);Ny=len(data['yh']);Nx=len(data['xh'])
    out_var[:,0]/=(Ny*Nx*Nt)
    out_var = np.array([out_var[:, 0], out_var[:, 2]]).T  # divides the data column by the counter column
    return out_var
    
def SBZTrackedVMFProfile(type,type2):

    if type2=='general':
        profile_array_VMF=VMF_G.copy() 
    elif type2=='cloudy':
        profile_array_VMF=VMF_C.copy() 

    if type=='all':
        out_arr=ALL_SBZ_out_arr
        # after_array=ALL_SBZ_out_after_array
    elif type=='shallow':
        out_arr=SHALLOW_SBZ_out_arr
        # after_array=SHALLOW_SBZ_out_after_array
    elif type=='deep':
        out_arr=DEEP_SBZ_out_arr
        # after_array=DEEP_SBZ_out_after_array
    
    
    zhs=data['zh'].values
    VMF_profile=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    VMF_profile[:,2]=zhs


    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust] #JOBARRAY INDEXADJUST
        ys=Y[ts,p-index_adjust] #JOBARRAY INDEXADJUST
        xs=X[ts,p-index_adjust] #JOBARRAY INDEXADJUST
        
        vars_VMF=profile_array_VMF[ts,p-index_adjust] #JOBARRAY INDEXADJUST
        np.add.at(VMF_profile[:, 0], zs, vars_VMF)
        np.add.at(VMF_profile[:, 1], zs, 1)
    
    print('done\n')
    return VMF_profile

In [61]:
# type2='general'
type2='cloudy'
SBZ_ALL_VMF = SBZTrackedVMFProfile(type='all',type2=type2)
SBZ_SHALLOW_VMF = SBZTrackedVMFProfile(type='shallow',type2=type2)
SBZ_DEEP_VMF = SBZTrackedVMFProfile(type='deep',type2=type2)

# Define the file path
filePath=dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
filePath+=f'SBZ_tracked_profiles_VMF_{res}_{t_res}_{Np_str}_{job_id}.h5'

# Write datasets to the HDF5 file
with h5py.File(filePath, 'w') as h5f:
    h5f.create_dataset('SBZ_ALL_VMF', data=SBZ_ALL_VMF)
    h5f.create_dataset('SBZ_SHALLOW_VMF', data=SBZ_SHALLOW_VMF)
    h5f.create_dataset('SBZ_DEEP_VMF', data=SBZ_DEEP_VMF)


done

done

done



In [62]:
#MAKING ENTRAINMENT PROFILE ARRAY FUNCTION

def averaged_profiles(profile):
    out_var = profile[(profile[:, 1] != 0)]  # gets rid of rows that have no data
    # out_var = np.array([out_var[:, 0] / out_var[:, 1], out_var[:, 2]]).T  # divides the data column by the counter column
    
    Nt=len(data['time']);Ny=len(data['yh']);Nx=len(data['xh'])
    out_var[:,0]/=(Ny*Nx*Nt)
    out_var = np.array([out_var[:, 0], out_var[:, 2]]).T  # divides the data column by the counter column
    return out_var
    
def nonSBZTrackedVMFProfile(type,type2):

    if type2=='general':
        profile_array_VMF=VMF_G.copy() 
    elif type2=='cloudy':
        profile_array_VMF=VMF_C.copy() 

    if type=='all':
        out_arr=ALL_nonSBZ_out_arr
        # after_array=ALL_nonSBZ_out_after_array
    elif type=='shallow':
        out_arr=SHALLOW_nonSBZ_out_arr
        # after_array=SHALLOW_nonSBZ_out_after_array
    elif type=='deep':
        out_arr=DEEP_nonSBZ_out_arr
        # after_array=DEEP_nonSBZ_out_after_array
    
    
    zhs=data['zh'].values
    VMF_profile=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    VMF_profile[:,2]=zhs


    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust] #JOBARRAY INDEXADJUST
        ys=Y[ts,p-index_adjust] #JOBARRAY INDEXADJUST
        xs=X[ts,p-index_adjust] #JOBARRAY INDEXADJUST
        
        vars_VMF=profile_array_VMF[ts,p-index_adjust] #JOBARRAY INDEXADJUST
        np.add.at(VMF_profile[:, 0], zs, vars_VMF)
        np.add.at(VMF_profile[:, 1], zs, 1)
    
    print('done\n')
    return VMF_profile

In [ ]:
# type2='general'
type2='cloudy'
nonSBZ_ALL_VMF = nonSBZTrackedVMFProfile(type='all',type2=type2)
nonSBZ_SHALLOW_VMF = nonSBZTrackedVMFProfile(type='shallow',type2=type2)
nonSBZ_DEEP_VMF = nonSBZTrackedVMFProfile(type='deep',type2=type2)

# Define the file path
filePath=dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
filePath+=f'nonSBZ_tracked_profiles_VMF_{res}_{t_res}_{Np_str}_{job_id}.h5'

# Write datasets to the HDF5 file
with h5py.File(filePath, 'w') as h5f:
    h5f.create_dataset('nonSBZ_ALL_VMF', data=nonSBZ_ALL_VMF)
    h5f.create_dataset('nonSBZ_SHALLOW_VMF', data=nonSBZ_SHALLOW_VMF)
    h5f.create_dataset('nonSBZ_DEEP_VMF', data=nonSBZ_DEEP_VMF)

In [103]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [109]:
if recombine==True:
    def get_profiles(job_id):
        # Define file path
        input_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
        input_file+=f'SBZ_tracked_profiles_VMF_{res}_{t_res}_{Np_str}_{job_id}.h5'
        
        # Load CL-tracked profiles and store them in a dictionary
        data_dict = {}
        
        with h5py.File(input_file, 'r') as h5f:
            for key in h5f.keys():
                data_dict[key] = h5f[key][:]  # Load each dataset into the dictionary
        return data_dict
    
    job_id=1
    dict1=get_profiles(job_id)
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
        dict2=get_profiles(job_id)
        for var in dict2:
            dict1[var][:,0:1+1]+=dict2[var][:,0:1+1]
    
    
    #SAVING INTO FINAL FORM
    output_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'+f'SBZ_tracked_profiles_VMF.h5'
    with h5py.File(output_file, 'w') as f:
        for var in dict1:
            profile_var = dict1[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

job_id = 10
job_id = 20
job_id = 30
job_id = 40
job_id = 50
job_id = 60


In [110]:
if recombine==True:
    def get_profiles(job_id):
        # Define file path
        input_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
        input_file+=f'nonSBZ_tracked_profiles_VMF_{res}_{t_res}_{Np_str}_{job_id}.h5'
        
        # Load CL-tracked profiles and store them in a dictionary
        data_dict = {}
        
        with h5py.File(input_file, 'r') as h5f:
            for key in h5f.keys():
                data_dict[key] = h5f[key][:]  # Load each dataset into the dictionary
        return data_dict
    
    job_id=1
    dict1=get_profiles(job_id)
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
        dict2=get_profiles(job_id)
        for var in dict2:
            dict1[var][:,0:1+1]+=dict2[var][:,0:1+1]
    
    
    #SAVING INTO FINAL FORM
    output_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'+f'nonSBZ_tracked_profiles_VMF.h5'
    with h5py.File(output_file, 'w') as f:
        for var in dict1:
            profile_var = dict1[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

job_id = 10
job_id = 20
job_id = 30
job_id = 40
job_id = 50
job_id = 60


In [72]:
#ColdPool
################################################################

In [75]:
#MAKING ENTRAINMENT PROFILE ARRAY FUNCTION

def averaged_profiles(profile):
    out_var = profile[(profile[:, 1] != 0)]  # gets rid of rows that have no data
    # out_var = np.array([out_var[:, 0] / out_var[:, 1], out_var[:, 2]]).T  # divides the data column by the counter column
    
    Nt=len(data['time']);Ny=len(data['yh']);Nx=len(data['xh'])
    out_var[:,0]/=(Ny*Nx*Nt)
    out_var = np.array([out_var[:, 0], out_var[:, 2]]).T  # divides the data column by the counter column
    return out_var
    
def ColdPoolTrackedVMFProfile(type,type2):

    if type2=='general':
        profile_array_VMF=VMF_G.copy() 
    elif type2=='cloudy':
        profile_array_VMF=VMF_C.copy() 

    if type=='all':
        out_arr=ALL_ColdPool_out_arr
        # after_array=ALL_ColdPool_after_array
    elif type=='shallow':
        out_arr=SHALLOW_ColdPool_out_arr
        # after_array=SHALLOW_ColdPool_after_array
    elif type=='deep':
        out_arr=DEEP_ColdPool_out_arr
        # after_array=DEEP_ColdPool_after_array
    
    
    zhs=data['zh'].values
    VMF_profile=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    VMF_profile[:,2]=zhs


    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust] #JOBARRAY INDEXADJUST
        ys=Y[ts,p-index_adjust] #JOBARRAY INDEXADJUST
        xs=X[ts,p-index_adjust] #JOBARRAY INDEXADJUST
        
        vars_VMF=profile_array_VMF[ts,p-index_adjust] #JOBARRAY INDEXADJUST
        np.add.at(VMF_profile[:, 0], zs, vars_VMF)
        np.add.at(VMF_profile[:, 1], zs, 1)
    
    print('done\n')
    return VMF_profile

In [76]:
# type2='general'
type2='cloudy'
ColdPool_ALL_VMF = ColdPoolTrackedVMFProfile(type='all',type2=type2)
ColdPool_SHALLOW_VMF = ColdPoolTrackedVMFProfile(type='shallow',type2=type2)
ColdPool_DEEP_VMF = ColdPoolTrackedVMFProfile(type='deep',type2=type2)

# Define the file path
filePath=dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
filePath+=f'ColdPool_tracked_profiles_VMF_{res}_{t_res}_{Np_str}_{job_id}.h5'

# Write datasets to the HDF5 file
with h5py.File(filePath, 'w') as h5f:
    h5f.create_dataset('ColdPool_ALL_VMF', data=ColdPool_ALL_VMF)
    h5f.create_dataset('ColdPool_SHALLOW_VMF', data=ColdPool_SHALLOW_VMF)
    h5f.create_dataset('ColdPool_DEEP_VMF', data=ColdPool_DEEP_VMF)

done

done

done



In [77]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [116]:
if recombine==True:
    def get_profiles(job_id):
        # Define file path
        input_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
        input_file+=f'ColdPool_tracked_profiles_VMF_{res}_{t_res}_{Np_str}_{job_id}.h5'
        
        # Load CL-tracked profiles and store them in a dictionary
        data_dict = {}
        
        with h5py.File(input_file, 'r') as h5f:
            for key in h5f.keys():
                data_dict[key] = h5f[key][:]  # Load each dataset into the dictionary
        return data_dict
    
    job_id=1
    dict1=get_profiles(job_id)
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
        dict2=get_profiles(job_id)
        for var in dict2:
            dict1[var][:,0:1+1]+=dict2[var][:,0:1+1]
    
    
    #SAVING INTO FINAL FORM
    output_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'+f'ColdPool_tracked_profiles_VMF_{res}_{t_res}_{Np_str}.h5'
    with h5py.File(output_file, 'w') as f:
        for var in dict1:
            profile_var = dict1[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

job_id = 10
job_id = 20
job_id = 30
job_id = 40
job_id = 50
job_id = 60
